# 04 — Optional reflection handling with UnReflectAnything

This notebook creates the optional **reflection-handled (`_ref`) input branch**. It reads only the already foreground-removed, scene-cropped outputs of notebook 03_A. View selection, crop, canvas size, padding, split, ordering, and foreground mask remain unchanged.

UnReflectAnything is an RGB-only **specular-highlight removal** model. A frozen DINOv3-Large encoder extracts image tokens; a reflection head predicts a soft highlight region; contaminated tokens are masked and filled by a learned token inpainter; a decoder reconstructs a diffuse-looking RGB image. Its synthetic training supervision uses MoGe-2 geometry with rendered Blinn–Phong/Fresnel highlights. It does not recover physically measured diffuse reflectance, and it may alter genuine paint/glass appearance. Therefore `_ref` is an ablation branch, not a replacement for the standard inputs.

Outputs are written separately under `data_processed/method_inputs_ref`. Notebook 03_A files are never overwritten. The official model internally resizes for inference and restores the original dimensions; afterward this notebook restores every pixel outside the foreground mask from the standard input and copies the original mask unchanged.


In [ ]:
%pip -q install "unreflectanything>=1.0.3"

import gc, subprocess
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU runtime before running notebook 04.")
print(torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path
from google.colab import drive

DRIVE_MOUNT = Path("/content/drive")
if not (DRIVE_MOUNT / "MyDrive").is_dir():
    drive.mount(str(DRIVE_MOUNT))
PROJECT_ROOT = DRIVE_MOUNT / "MyDrive" / "ITU" / "3D" / "Thesis"


In [ ]:
import subprocess, sys
CODE_ROOT = Path("/content/Project_Thesis_code")
REPOSITORY = "https://github.com/katlit/Project_Thesis.git"
BRANCH = "codex/hq200-example-notebook"
if not (CODE_ROOT / "code" / "src").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY, str(CODE_ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(CODE_ROOT), "pull", "--ff-only"], check=True)
sys.path.insert(0, str(CODE_ROOT / "code")) if str(CODE_ROOT / "code") not in sys.path else None


In [ ]:
from src.reflection_preprocessing import (
    process_with_unreflectanything, reflection_difference_image,
)
import unreflectanything

INPUT_ROOT = PROJECT_ROOT / "data_processed" / "method_inputs"
OUTPUT_ROOT = PROJECT_ROOT / "data_processed" / "method_inputs_ref"
INPUT_MANIFEST = INPUT_ROOT / "manifest.csv"

METHODS_TO_PROCESS = ["3dgs", "dust3r_mast3r", "vggt"]
DATASETS_TO_PROCESS = None       # e.g. ["3DRealCar"] or None for all
RUN_EXPORT = False              # inspect the preview first, then set True
OVERWRITE = False               # False safely resumes an interrupted export
PREVIEW_IMAGES = 1             # raise only after one image succeeds
HIGHLIGHT_THRESHOLD = 0.30       # lower finds more suspected highlights
MASK_DILATION = 40               # context removed around predicted highlights
USE_MIXED_PRECISION = True       # substantially lowers activation memory

if not INPUT_MANIFEST.is_file():
    raise FileNotFoundError(f"Run notebook 03_A export first: {INPUT_MANIFEST}")
manifest = pd.read_csv(INPUT_MANIFEST)
selected = manifest[manifest.method.isin(METHODS_TO_PROCESS)].copy()
if DATASETS_TO_PROCESS is not None:
    selected = selected[selected.dataset.isin(DATASETS_TO_PROCESS)]
selected = selected.sort_values(["method", "dataset", "scene", "split", "view_order", "source"], na_position="last")
print("Rows:", len(selected), "| scenes:", selected[["dataset", "scene"]].drop_duplicates().shape[0])
display(selected[["method", "dataset", "scene", "split", "view_order", "method_image", "method_mask"]].head(12))


## Load the released model

The weights are downloaded once into Colab's package cache. `threshold` and `dilation` are the two official inference controls. Start with the defaults; compare alternatives on several glossy, glass, and low-highlight views before a full export.


In [ ]:
weights = Path(unreflectanything.cache("weights"))
if not any(weights.glob("*.pth")):
    subprocess.run(["unreflectanything", "download", "--weights"], check=True)
free_before, total_memory = torch.cuda.mem_get_info()
print(f"Before model: {free_before / 1024**3:.1f}/{total_memory / 1024**3:.1f} GiB free")
model = unreflectanything.model(pretrained=True, device=torch.device("cuda"), verbose=False)
model.eval().requires_grad_(False)
free_after, _ = torch.cuda.mem_get_info()
print(f"After model:  {free_after / 1024**3:.1f}/{total_memory / 1024**3:.1f} GiB free")
print("Weights:", weights)


In [ ]:
# A temporary preview does not write to Drive.
preview_root = Path("/content/unreflectanything_preview")
preview_rows = selected.query("method == '3dgs' and split == 'train'").head(PREVIEW_IMAGES)
preview_records = []
for row in tqdm(preview_rows.itertuples(index=False), total=len(preview_rows), desc="Preview"):
    out_image = preview_root / row.dataset / row.scene / Path(row.method_image).name
    out_mask = preview_root / row.dataset / row.scene / "masks" / Path(row.method_mask).name
    report = process_with_unreflectanything(
        row.method_image, row.method_mask, out_image, out_mask, model,
        threshold=HIGHLIGHT_THRESHOLD, dilation=MASK_DILATION, overwrite=True,
        use_amp=USE_MIXED_PRECISION,
    )
    preview_records.append((row, out_image, report))

fig, axes = plt.subplots(len(preview_records), 3, figsize=(12, 4 * len(preview_records)), squeeze=False)
for r, (row, output_path, report) in enumerate(preview_records):
    before = Image.open(row.method_image).convert("RGB")
    after = Image.open(output_path).convert("RGB")
    axes[r, 0].imshow(before); axes[r, 0].set_title("Standard cropped input")
    axes[r, 1].imshow(after); axes[r, 1].set_title("Reflection handled")
    axes[r, 2].imshow(reflection_difference_image(before, after)); axes[r, 2].set_title("|change| × 4 (diagnostic)")
    axes[r, 0].set_ylabel(f"{row.dataset}\n{row.scene}\nMAE {report['foreground_mae']:.3f}")
    for axis in axes[r]: axis.axis("off")
plt.tight_layout(); plt.show()


## Export the `_ref` branch

Set `RUN_EXPORT=True` only after the preview is acceptable. `OVERWRITE=False` resumes safely and retains finished files. The output manifest keeps all source metadata, adds provenance/settings/change diagnostics, and changes only `method_image` and `method_mask` to the parallel paths.


In [ ]:
records = []
if RUN_EXPORT:
    for row in tqdm(selected.itertuples(index=False), total=len(selected), desc="Reflection handling"):
        relative_image = Path(row.method_image).relative_to(INPUT_ROOT)
        relative_mask = Path(row.method_mask).relative_to(INPUT_ROOT)
        output_image = OUTPUT_ROOT / relative_image
        output_mask = OUTPUT_ROOT / relative_mask
        report = process_with_unreflectanything(
            row.method_image, row.method_mask, output_image, output_mask, model,
            threshold=HIGHLIGHT_THRESHOLD, dilation=MASK_DILATION, overwrite=OVERWRITE,
            use_amp=USE_MIXED_PRECISION,
        )
        record = row._asdict()
        record.update({
            "source_method_image": record["method_image"],
            "source_method_mask": record["method_mask"],
            "method_image": str(output_image), "method_mask": str(output_mask),
            "input_variant": "reflection_handled", "reflection_model": "UnReflectAnything",
            "reflection_threshold": HIGHLIGHT_THRESHOLD, "reflection_dilation": MASK_DILATION,
            **report,
        })
        records.append(record)
    output_manifest = pd.DataFrame(records)
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    output_manifest.to_csv(OUTPUT_ROOT / "manifest.csv", index=False)
    print("Saved:", OUTPUT_ROOT / "manifest.csv")
    display(output_manifest.groupby(["method", "dataset"])[["foreground_mae", "changed_foreground_fraction"]].agg(["mean", "median"]).round(4))
else:
    print("Preview only. Set RUN_EXPORT=True to create the Drive outputs.")


In [ ]:
del model
gc.collect(); torch.cuda.empty_cache()
print("GPU memory released.")


## What to inspect

Look for removal of compact white glare without loss of lamps, window borders, logos, or paint color. The amplified difference is only a change map—not the model's highlight mask and not a quality score. If real structure disappears, increase `HIGHLIGHT_THRESHOLD`, reduce `MASK_DILATION`, or keep the standard branch for that experiment.
